In [1]:
import random
import re

import numpy as np

from greek_nlp.rag.retriever import Retriever

r = Retriever.load()
docs = r.docs
print(len(docs), "άρθρα στο ευρετήριο")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

5250 άρθρα στο ευρετήριο


In [2]:
random.seed(42)
SENT_SPLIT = re.compile(r"(?<=[.!?;])\s+")


def lead_query(text, n_words=20):
    return " ".join(text.split()[:n_words])


def late_query(text, min_words=12):
    words = text.split()
    if len(words) < 260:  # μόνο αρκετά μεγάλα άρθρα
        return None
    tail = " ".join(words[200:])  # μέρος εκτός των πρώτων 256 tokens
    sents = [s for s in SENT_SPLIT.split(tail) if len(s.split()) >= min_words]
    return random.choice(sents) if sents else None


order = list(range(len(docs)))
random.shuffle(order)

lead_q, lead_gold, late_q, late_gold = [], [], [], []
for i in order:
    text = docs[i]["text"]
    if len(lead_q) < 300:
        lead_q.append(lead_query(text))
        lead_gold.append(i)
    if len(late_q) < 300:
        q = late_query(text)
        if q:
            late_q.append(q)
            late_gold.append(i)
    if len(lead_q) == 300 and len(late_q) == 300:
        break

print(len(lead_q), len(late_q))
print("Παράδειγμα late:", late_q[0])

300 300
Παράδειγμα late: Για την ώρα όμως ο ΑΠΟΕΛ δεν επισημοποιεί το ενδιαφέρον του, αφού προηγείτο η πρόσληψη προπονητή.


In [3]:
def evaluate(queries, gold, k=10):
    emb = r.model.encode(
        ["query: " + q for q in queries], normalize_embeddings=True, batch_size=32
    ).astype(np.float32)
    _, ids = r.index.search(emb, k)
    ranks = []
    for row, g in zip(ids, gold):
        hits = np.where(row == g)[0]
        ranks.append(int(hits[0]) + 1 if len(hits) else None)
    out = {}
    for cut in (1, 5, 10):
        out[f"Recall@{cut}"] = np.mean([rk is not None and rk <= cut for rk in ranks])
    out["MRR@10"] = np.mean([1 / rk if rk else 0 for rk in ranks])
    return out


for name, q, g in [("lead", lead_q, lead_gold), ("late", late_q, late_gold)]:
    res = evaluate(q, g)
    print(name, {k: round(float(v), 3) for k, v in res.items()})

lead {'Recall@1': 0.997, 'Recall@5': 1.0, 'Recall@10': 1.0, 'MRR@10': 0.998}
late {'Recall@1': 0.327, 'Recall@5': 0.463, 'Recall@10': 0.53, 'MRR@10': 0.386}
